# Hyperparameter Tuning — Random Forest URL Classifier
**Google Colab | Dataset 200.000 Balanced | RandomizedSearchCV + StratifiedKFold**

---
Notebook ini mencari kombinasi hyperparameter terbaik untuk model Random Forest
menggunakan **RandomizedSearchCV** (lebih efisien dari GridSearch untuk ruang besar).

Output notebook ini adalah file `best_params.json` yang langsung dipakai di notebook training.

**Alur:**
1. Load dataset → ambil 200.000 sampel balanced (100K aman + 100K porno)
2. Definisikan ruang pencarian hyperparameter
3. Jalankan RandomizedSearchCV (30 iterasi × 3-fold CV)
4. Simpan hyperparameter terbaik ke `best_params.json`

## Cell 1 — Install Library

In [ ]:
# Semua library yang dibutuhkan sudah tersedia di Colab,
# kecuali seaborn yang kadang perlu diupdate.
!pip install -q --upgrade scikit-learn seaborn
print('Library siap.')

## Cell 2 — Import Library

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import time
import os
import datetime
import warnings
warnings.filterwarnings('ignore')

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV, StratifiedKFold
)
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    make_scorer, classification_report
)
from sklearn.utils.class_weight import compute_class_weight

print('Library berhasil diimport.')
print(f'scikit-learn version: {__import__("sklearn").__version__}')

## Cell 3 — Mount Google Drive & Konfigurasi Path
Sesuaikan `DATASET_PATH` dengan lokasi file CSV dataset kamu di Google Drive.

**Format dataset yang diharapkan** — CSV dengan kolom:
```
url, label, domain_length, digit_count, dot_count, delimiter_count,
suspicious_word_count, digit_letter_ratio, max_sequential_digits
```
> `label`: 0 = URL aman, 1 = URL pornografi

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ================================================================
# SESUAIKAN PATH INI
# ================================================================
DATASET_PATH = '/content/drive/MyDrive/Tugas Akhir/Dataset/dataset_url.csv'
SAVE_PATH    = '/content/drive/MyDrive/Tugas Akhir/Model/RF/'
# ================================================================

os.makedirs(SAVE_PATH, exist_ok=True)

FEATURE_COLS = [
    'domain_length',
    'digit_count',
    'dot_count',
    'delimiter_count',
    'suspicious_word_count',
    'digit_letter_ratio',
    'max_sequential_digits'
]
LABEL_COL = 'label'

df = pd.read_csv(DATASET_PATH)
print(f'Dataset berhasil dimuat.')
print(f'Shape : {df.shape[0]:,} baris x {df.shape[1]} kolom')
print(f'Kolom : {list(df.columns)}')
print(f'\nDistribusi label:')
print(df[LABEL_COL].value_counts())
df.head()

## Cell 4 — Sampling 200.000 Data Balanced

Kenapa 200.000?
- Random Forest dengan 7 fitur mencapai plateau performa di sekitar angka ini
- Cukup untuk CNN-1D menunjukkan kemampuannya
- Dataset yang sama dipakai untuk **kedua model** agar perbandingan adil

Jika dataset asli kurang dari 100K per kelas, sesuaikan `N_PER_CLASS`.

In [ ]:
N_PER_CLASS = 100_000  # 100K aman + 100K porno = 200K total

df_safe = df[df[LABEL_COL] == 0]
df_porn = df[df[LABEL_COL] == 1]

print(f'Total URL aman      : {len(df_safe):,}')
print(f'Total URL pornografi: {len(df_porn):,}')

# Pastikan jumlah data cukup
assert len(df_safe) >= N_PER_CLASS, f'Data aman kurang dari {N_PER_CLASS:,}'
assert len(df_porn) >= N_PER_CLASS, f'Data porno kurang dari {N_PER_CLASS:,}'

df_safe_s = df_safe.sample(n=N_PER_CLASS, random_state=42)
df_porn_s = df_porn.sample(n=N_PER_CLASS, random_state=42)

# Gabung dan acak
df_bal = pd.concat([df_safe_s, df_porn_s]).sample(frac=1, random_state=42).reset_index(drop=True)

# Bersihkan duplikat dan missing value
n_before = len(df_bal)
df_bal = df_bal.drop_duplicates(subset=FEATURE_COLS + [LABEL_COL])
df_bal = df_bal.dropna(subset=FEATURE_COLS + [LABEL_COL]).reset_index(drop=True)
print(f'\nSetelah sampling   : {n_before:,} baris')
print(f'Setelah bersih     : {len(df_bal):,} baris')
print(f'Distribusi akhir   :\n{df_bal[LABEL_COL].value_counts()}')
df_bal[FEATURE_COLS].describe()

## Cell 5 — Split Data Training & Testing

80% untuk training (dipakai CV), 20% untuk evaluasi akhir.
`stratify=y` memastikan proporsi kelas tetap sama di train dan test.

In [ ]:
X = df_bal[FEATURE_COLS].values.astype(np.float32)
y = df_bal[LABEL_COL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Training set : {len(X_train):,} baris')
print(f'Testing set  : {len(X_test):,} baris')
print(f'Bentuk input : {X_train.shape}  → {X_train.shape[1]} fitur')

# Distribusi kelas per split
print(f'\nDistribusi Kelas per Split:')
for split_name, y_split in [('Train', y_train), ('Test', y_test)]:
    n_aman  = int((y_split == 0).sum())
    n_porno = int((y_split == 1).sum())
    total   = len(y_split)
    print(f'  {split_name:<8}: aman={n_aman:,}, porno={n_porno:,} ({n_porno/total*100:.1f}%)')

print(f'\nDistribusi train: {np.bincount(y_train)}')
print(f'Distribusi test : {np.bincount(y_test)}')

## Cell 6 — Definisi Ruang Pencarian Hyperparameter

Penjelasan setiap hyperparameter yang dituning (sesuai Tabel III.2 Laporan TA):

| Parameter | Nilai yang Diuji | Pengaruh |
|-----------|-----------------|----------|
| `n_estimators` | 50, 100, 200, 300, 400, 500 | Jumlah pohon. Lebih banyak = lebih stabil, tapi lebih lambat |
| `max_depth` | None, 5, 10, 15, 20, 25, 30 | Kedalaman pohon. `None` = tanpa batas (risiko overfitting) |
| `max_features` | None, sqrt, log2 | Jumlah fitur yang dipertimbangkan setiap split |

Total kombinasi: 6 × 7 × 3 = **90 kombinasi**

RandomizedSearchCV akan mencoba **30 iterasi** acak dari 90 kombinasi tersebut,
lebih efisien dari Grid Search yang harus mencoba semua 90 kombinasi.

In [ ]:
param_dist = {
    'n_estimators': [50, 100, 200, 300, 400, 500],
    'max_depth'   : [None, 5, 10, 15, 20, 25, 30],
    'max_features': [None, 'sqrt', 'log2'],
}

# Hitung total kombinasi
total_kombinasi = 1
for v in param_dist.values():
    total_kombinasi *= len(v)

print('Ruang pencarian hyperparameter (sesuai Tabel III.2 Laporan TA):')
for k, v in param_dist.items():
    print(f'  {k:<22}: {v}')
print(f'\nTotal kombinasi : {total_kombinasi:,}  (6 × 7 × 3 = 90)')
print(f'Yang dicoba     : 30 iterasi (RandomizedSearchCV)')
print(f'Cross-validation: 3-fold StratifiedKFold')

## Cell 7 — Jalankan RandomizedSearchCV

**Metrik optimasi: Accuracy** (sesuai laporan TA, halaman 987 — pemilihan model terbaik
berdasarkan validation accuracy tertinggi).

Dataset yang balanced (50:50 aman:porno) membuat accuracy menjadi metrik yang
representatif dan konsisten dengan kriteria seleksi hyperparameter terbaik pada CNN-1D.

> Estimasi waktu: **15–40 menit** tergantung hardware Colab yang didapat.

In [ ]:
TIMESTAMP        = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
TUNING_PLOTS_DIR = SAVE_PATH + f'RF_Tuning_Plots_{TIMESTAMP}/'
os.makedirs(TUNING_PLOTS_DIR, exist_ok=True)

# Multi-scoring: evaluasi semua metrik utama per skenario
scoring = {
    'accuracy' : make_scorer(accuracy_score),
    'f1'       : make_scorer(f1_score,        zero_division=0),
    'precision': make_scorer(precision_score, zero_division=0),
    'recall'   : make_scorer(recall_score,    zero_division=0),
}

cv      = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)

search = RandomizedSearchCV(
    estimator           = rf_base,
    param_distributions = param_dist,
    n_iter              = 30,
    cv                  = cv,
    scoring             = scoring,
    refit               = 'accuracy',   # pilih best model berdasarkan accuracy
    n_jobs              = -1,
    verbose             = 1,
    random_state        = 42,
    return_train_score  = True
)

print('Memulai RandomizedSearchCV...')
print(f'(30 iterasi × 3-fold = 90 kali training, setiap kali dengan ~{len(X_train)*2//3:,} data)')
print(f'Metrik       : accuracy + f1 + precision + recall (semua dievaluasi via CV)')
print(f'Plot per exp : {TUNING_PLOTS_DIR}')
print('-' * 55)

t0 = time.time()
search.fit(X_train, y_train)
durasi = time.time() - t0

print(f'\nSelesai dalam {durasi:.0f} detik ({durasi/60:.1f} menit)')
print(f'\nAccuracy terbaik (CV mean) : {search.best_score_:.4f}')
print(f'Hyperparameter terbaik:')
for k, v in search.best_params_.items():
    print(f'  {k:<22}: {v}')

## Cell 8 — Tampilkan Hasil Semua Iterasi

In [ ]:
# Bangun DataFrame hasil semua skenario tuning dengan semua metrik CV (multi-scoring)
results_raw = pd.DataFrame(search.cv_results_)
param_cols  = sorted([c for c in results_raw.columns if c.startswith('param_')])

rows = []
for _, row in results_raw.iterrows():
    r = {}
    for col in param_cols:
        r[col.replace('param_', '')] = row[col]
    r['accuracy_cv_mean']    = round(float(row['mean_test_accuracy']),  4)
    r['accuracy_cv_std']     = round(float(row['std_test_accuracy']),   4)
    r['f1_cv_mean']          = round(float(row['mean_test_f1']),        4)
    r['precision_cv_mean']   = round(float(row['mean_test_precision']), 4)
    r['recall_cv_mean']      = round(float(row['mean_test_recall']),    4)
    r['accuracy_train_mean'] = round(float(row['mean_train_accuracy']), 4)
    r['waktu_fit_detik']     = round(float(row['mean_fit_time']),       2)
    rows.append(r)

results_df = pd.DataFrame(rows)
results_df = results_df.sort_values('accuracy_cv_mean', ascending=False).reset_index(drop=True)
results_df.insert(0, 'no_eksperimen', range(1, len(results_df) + 1))

print(f'Total skenario : {len(results_df)}')
print(f'Best accuracy  : {results_df["accuracy_cv_mean"].iloc[0]:.4f}  (Skenario 1)')
print(f'Worst accuracy : {results_df["accuracy_cv_mean"].iloc[-1]:.4f}  (Skenario {len(results_df)})')
print()
display(results_df[['no_eksperimen', 'n_estimators', 'max_depth', 'max_features',
                     'accuracy_cv_mean', 'f1_cv_mean', 'precision_cv_mean', 'recall_cv_mean',
                     'accuracy_train_mean', 'waktu_fit_detik']])

# ── Per-experiment subfolders ─────────────────────────────────────
print(f'\nMenyimpan per-experiment plots ke {TUNING_PLOTS_DIR}...')
for i, row in results_df.iterrows():
    exp_id   = int(row['no_eksperimen'])
    n_est    = row['n_estimators']
    max_d    = row['max_depth']
    max_f    = row['max_features']
    exp_name = f'nEst{n_est}_maxD{max_d}_maxF{max_f}'
    exp_folder = TUNING_PLOTS_DIR + f'exp{exp_id:02d}_{exp_name}/'
    os.makedirs(exp_folder, exist_ok=True)

    # metrics_barchart.png
    fig, ax = plt.subplots(figsize=(6, 4))
    m_names  = ['Accuracy', 'F1-Score', 'Precision', 'Recall']
    m_values = [row['accuracy_cv_mean'], row['f1_cv_mean'],
                row['precision_cv_mean'], row['recall_cv_mean']]
    bar_colors = ['#1E88E5', '#43A047', '#FB8C00', '#E53935']
    bars = ax.bar(m_names, m_values, color=bar_colors, alpha=0.85)
    for bar, val in zip(bars, m_values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{val:.4f}', ha='center', va='bottom', fontsize=9)
    ax.set_ylim(0, 1.1)
    ax.set_title(f'CV Metrics — Exp {exp_id:02d}: {exp_name}', fontsize=9)
    ax.set_ylabel('Skor (CV Mean)')
    ax.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(exp_folder + 'metrics_barchart.png', dpi=120, bbox_inches='tight')
    plt.close()

    # experiment_info.json
    exp_info = {
        'experiment_id'  : exp_id,
        'name'           : exp_name,
        'hyperparameters': {
            'n_estimators': int(n_est),
            'max_depth'   : None if (max_d is None or str(max_d) == 'None') else int(max_d),
            'max_features': str(max_f) if max_f is not None else None,
        },
        'cv_metrics': {
            'accuracy_cv_mean'   : float(row['accuracy_cv_mean']),
            'f1_cv_mean'         : float(row['f1_cv_mean']),
            'precision_cv_mean'  : float(row['precision_cv_mean']),
            'recall_cv_mean'     : float(row['recall_cv_mean']),
            'accuracy_cv_std'    : float(row['accuracy_cv_std']),
            'accuracy_train_mean': float(row['accuracy_train_mean']),
        },
        'waktu_fit_detik': float(row['waktu_fit_detik']),
    }
    with open(exp_folder + 'experiment_info.json', 'w') as f:
        json.dump(exp_info, f, indent=2)

print(f'✅ {len(results_df)} per-experiment subfolder tersimpan.')

# ── Horizontal comparison chart ───────────────────────────────────
sorted_for_plot = results_df.sort_values('accuracy_cv_mean', ascending=True)
labels_plot = [f"nEst{r['n_estimators']}_maxD{r['max_depth']}_maxF{r['max_features']}"
               for _, r in sorted_for_plot.iterrows()]
vals_plot   = (sorted_for_plot['accuracy_cv_mean'] * 100).tolist()
best_acc_cv = float(results_df['accuracy_cv_mean'].max())
colors_plot = ['#D32F2F' if abs(v/100 - best_acc_cv) < 1e-6 else '#1E88E5' for v in vals_plot]

fig_h = max(6, len(sorted_for_plot) * 0.45)
fig, ax = plt.subplots(figsize=(12, fig_h))
bars = ax.barh(range(len(labels_plot)), vals_plot, color=colors_plot, alpha=0.85)
ax.set_yticks(range(len(labels_plot)))
ax.set_yticklabels(labels_plot, fontsize=8)
ax.set_xlabel('Accuracy CV Mean (%)')
ax.set_title('Perbandingan Accuracy Semua Skenario RF Tuning\n(Merah = Terbaik)')
ax.axvline(best_acc_cv * 100, color='red', linestyle='--', alpha=0.5,
           label=f'Terbaik: {best_acc_cv:.4f}')
for bar, val in zip(bars, vals_plot):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}%', va='center', fontsize=7)
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
comparison_path = TUNING_PLOTS_DIR + 'rf_tuning_comparison.png'
plt.savefig(comparison_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ Plot perbandingan tersimpan: {comparison_path}')

## Cell 9 — Evaluasi Model Terbaik pada Test Set

In [ ]:
best_model = search.best_estimator_
y_pred = best_model.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec  = recall_score(y_test, y_pred, zero_division=0)
f1   = f1_score(y_test, y_pred, zero_division=0)

print('Evaluasi pada TEST SET (20% data yang tidak digunakan saat tuning):')
print('=' * 55)
print(f'  Accuracy  : {acc:.4f}')
print(f'  Precision : {prec:.4f}')
print(f'  Recall    : {rec:.4f}')
print(f'  F1-Score  : {f1:.4f}')
print('=' * 55)
print(classification_report(y_test, y_pred,
      target_names=['Aman (0)', 'Pornografi (1)'], digits=4))

# Cek overfitting
y_pred_train = best_model.predict(X_train)
acc_train = accuracy_score(y_train, y_pred_train)
gap = acc_train - acc
print(f'Cek Overfitting (Accuracy):')
print(f'  Acc Train : {acc_train:.4f}')
print(f'  Acc Test  : {acc:.4f}')
print(f'  Gap       : {gap:.4f}', end=' ')
if gap < 0.02:
    print('✅ Baik (tidak overfitting)')
elif gap < 0.05:
    print('⚠️ Sedikit overfitting')
else:
    print('❌ Overfitting — coba kurangi max_depth')

# Bar chart metrik test set
fig, ax = plt.subplots(figsize=(6, 4))
m_names  = ['Accuracy', 'F1-Score', 'Precision', 'Recall']
m_values = [acc, f1, prec, rec]
bar_colors = ['#1E88E5', '#43A047', '#FB8C00', '#E53935']
bars = ax.bar(m_names, m_values, color=bar_colors, alpha=0.85)
for bar, val in zip(bars, m_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.4f}', ha='center', va='bottom', fontsize=10)
ax.set_ylim(0, 1.1)
ax.set_title('Metrik Test Set — Best RF Model')
ax.set_ylabel('Skor')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(TUNING_PLOTS_DIR + 'best_model_test_metrics.png', dpi=120, bbox_inches='tight')
plt.show()

## Cell 10 — Simpan Hyperparameter Terbaik ke JSON
File ini akan dibaca otomatis oleh notebook training.

In [ ]:
best_params = search.best_params_.copy()

# Simpan best_params ke JSON
json_path = SAVE_PATH + 'best_params.json'
with open(json_path, 'w') as f:
    json.dump(best_params, f, indent=2, default=str)

print(f'✅ best_params.json tersimpan: {json_path}')
print('\nIsi best_params.json:')
print(json.dumps(best_params, indent=2, default=str))

# Simpan tabel semua skenario ke CSV
csv_path = SAVE_PATH + 'rf_tuning_results.csv'
results_df.to_csv(csv_path, index=False)
print(f'\n✅ CSV tersimpan : {csv_path}')
print(f'   Jumlah skenario : {len(results_df)}')
print(f'   Kolom           : {list(results_df.columns)}')

# ── Generate resume_tuning_rf.md ──────────────────────────────────
_n_exp    = len(results_df)
_best_row = results_df.iloc[0]
_bp       = search.best_params_
_best_acc = float(results_df['accuracy_cv_mean'].max())

ranking_rows = ''
for _, row in results_df.iterrows():
    rank   = int(row['no_eksperimen'])
    marker = ' ★' if rank == 1 else ''
    ranking_rows += (
        f'| {rank} | {row["n_estimators"]} | {row["max_depth"]} | {row["max_features"]} | '
        f'{row["accuracy_cv_mean"]:.4f} | {row["f1_cv_mean"]:.4f} | '
        f'{row["precision_cv_mean"]:.4f} | {row["recall_cv_mean"]:.4f} | '
        f'{row["waktu_fit_detik"]:.1f}s |{marker}\n'
    )

resume_md = (
    "# Resume Hyperparameter Tuning — Random Forest URL Classifier\n\n"
    "## 1. Konfigurasi Tuning\n"
    "- **Metode**: RandomizedSearchCV + StratifiedKFold\n"
    "- **Jumlah iterasi**: 30\n"
    "- **Cross-validation**: 3-fold Stratified\n"
    "- **Metrik optimasi**: Accuracy (refit='accuracy')\n"
    "- **Metrik dievaluasi**: Accuracy, F1, Precision, Recall (multi-scoring)\n"
    f"- **Total skenario dicoba**: {_n_exp}\n\n"
    "## 2. Ruang Pencarian Hyperparameter\n"
    "| Parameter | Nilai |\n"
    "|-----------|-------|\n"
    "| `n_estimators` | [50, 100, 200, 300, 400, 500] |\n"
    "| `max_depth` | [None, 5, 10, 15, 20, 25, 30] |\n"
    "| `max_features` | [None, 'sqrt', 'log2'] |\n\n"
    "## 3. Hyperparameter Terbaik\n"
    f"- **n_estimators** : {_bp.get('n_estimators')}\n"
    f"- **max_depth**    : {_bp.get('max_depth')}\n"
    f"- **max_features** : {_bp.get('max_features')}\n"
    f"- **Accuracy CV mean**: {_best_acc:.4f}\n\n"
    "## 4. Ranking Semua Skenario\n"
    "| Rank | n_estimators | max_depth | max_features | Acc | F1 | Precision | Recall | Waktu |\n"
    "|------|-------------|-----------|--------------|-----|----|-----------|--------|-------|\n"
    + ranking_rows +
    "\n## 5. Hasil Evaluasi Test Set\n"
    f"- **Accuracy**  : {acc:.4f}\n"
    f"- **F1-Score**  : {f1:.4f}\n"
    f"- **Precision** : {prec:.4f}\n"
    f"- **Recall**    : {rec:.4f}\n\n"
    "## 6. Output\n"
    "- **best_params.json** : hyperparameter terbaik untuk training final\n"
    "- **rf_tuning_results.csv** : tabel lengkap semua skenario\n"
    "- **rf_tuning_comparison.png** : horizontal bar chart perbandingan\n"
    f"- **exp01_../ ... exp{_n_exp:02d}_../** : per-experiment subfolder (metrics_barchart.png + experiment_info.json)\n\n"
    "> Hyperparameter ini digunakan di notebook `rf_training_v2.ipynb` untuk training final dengan 200K data.\n"
)

resume_path = SAVE_PATH + 'resume_tuning_rf.md'
with open(resume_path, 'w', encoding='utf-8') as f:
    f.write(resume_md)
print(f'\n✅ Resume Markdown tersimpan: {resume_path}')
print('\n--- SELESAI ---')
print('Langkah berikutnya: Buka notebook rf_training_v2.ipynb dengan best_params.json')